# Build Driver Standings

#### Sources
1. fact_session_results
1. dim_drivers

#### Output Columns
1. season
2. driver id
3. driver name
4. nationality
4. race starts
6. total points
7. number of wins
8. number of podiums
9. standing position

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_driver_standing AS
WITH driver_session_summary AS (
  select
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality,
    count(*) as race_starts,
    sum(r.points) as total_points,
    count_if(r.is_win) as number_of_wins,
    count_if(r.is_podium) as number_of_podiums
  from
    formula1.gold.fact_session_results r
      join formula1.gold.dim_drivers d
        on r.driver_id = d.driver_id
  GROUP BY
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality
)
SELECT
  season,
  driver_id,
  driver_name,
  nationality,
  RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
  race_starts,
  total_points,
  number_of_wins,
  number_of_podiums
FROM
  driver_session_summary